# Ship Detection

This notebook demonstrates how to use the geoai package for ship detection using a pre-trained model.

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/ship_detection.ipynb)

## Install package
To use the `geoai-py` package, ensure it is installed in your environment. Uncomment the command below if needed.

In [ ]:
%pip install -q geoai-py rioxarray

## Import libraries

In [ ]:
import geoai
import geopandas as gpd
import matplotlib.pyplot as plt
import rioxarray as rxr

## Download sample data

The clipped Dubai ship scene is hosted in the course GitHub repository.

In [ ]:
raster_url = (
    "https://raw.githubusercontent.com/nilaynishant/acrs/main/Data/ships_dubai_clipped.tif"
)
raster_path = geoai.download_file(raster_url, "ships_dubai_clipped.tif")

## Visualize data

In [ ]:
da = rxr.open_rasterio(raster_path, masked=True)  # dims: band, y, x
rgb = da.sel(band=[1, 2, 3])


def show_rgb(ax, title=None):
    """Draw the RGB image (2-98 % stretch) on a matplotlib axis."""
    rgb.plot.imshow(ax=ax, robust=True)
    ax.set_title(title)
    ax.set_aspect("equal")
    return ax


fig, ax = plt.subplots(figsize=(10, 10))
show_rgb(ax, "Ships – Dubai (input image)")
plt.tight_layout()
plt.show()

## Initialize model

In [ ]:
detector = geoai.ShipDetector()

## Generate masks

In [ ]:
output_path = "ships_dubai_masks.tif"

In [ ]:
masks_path = detector.generate_masks(
    raster_path,
    output_path=output_path,
    confidence_threshold=0.9,
    mask_threshold=0.7,
    overlap=0.25,
    chip_size=(256, 256),
    batch_size=4,
)

## Vectorize masks

In [ ]:
gdf = detector.vectorize_masks(
    output_path,
    output_path="ships_dubai_masks.geojson",
    confidence_threshold=0.8,
    min_object_area=100,
    max_object_size=10000,
)

## Visualize initial results

In [ ]:
gdf_plot = gdf.to_crs(da.rio.crs)

fig, ax = plt.subplots(figsize=(10, 10))
show_rgb(ax, "Detected ships (initial results)")
gdf_plot.plot(
    ax=ax,
    column="confidence",
    cmap="plasma",
    alpha=0.5,
    edgecolor="cyan",
    linewidth=0.8,
    legend=True,
    legend_kwds={"label": "Confidence", "shrink": 0.7},
)
plt.tight_layout()
plt.show()

## Calculate geometric properties

In [ ]:
gdf = geoai.add_geometric_properties(gdf)
gdf.head()

In [ ]:
gdf_plot = gdf.to_crs(da.rio.crs)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

show_rgb(axes[0], "Ships – confidence")
gdf_plot.plot(
    ax=axes[0],
    column="confidence",
    cmap="plasma",
    alpha=0.6,
    edgecolor="cyan",
    linewidth=0.8,
    legend=True,
    legend_kwds={"label": "Confidence", "shrink": 0.7},
)

show_rgb(axes[1], "Ships – elongation")
gdf_plot.plot(
    ax=axes[1],
    column="elongation",
    cmap="RdYlGn_r",
    alpha=0.6,
    edgecolor="white",
    linewidth=0.6,
    legend=True,
    legend_kwds={"label": "Elongation", "shrink": 0.7},
)

plt.tight_layout()
plt.show()

## Filter results

In [ ]:
aoi = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {},
            "geometry": {
                "type": "Polygon",
                "coordinates": [
                    [
                        [55.133729, 25.110277],
                        [55.134072, 25.11393],
                        [55.134823, 25.115601],
                        [55.136025, 25.117116],
                        [55.137677, 25.118127],
                        [55.140145, 25.118787],
                        [55.142248, 25.11902],
                        [55.142012, 25.118243],
                        [55.140831, 25.116728],
                        [55.13948, 25.116903],
                        [55.137956, 25.116825],
                        [55.136132, 25.115543],
                        [55.13566, 25.114416],
                        [55.135467, 25.1136],
                        [55.135939, 25.112609],
                        [55.136218, 25.111657],
                        [55.13551, 25.110685],
                        [55.134373, 25.110102],
                        [55.133729, 25.110277],
                    ]
                ],
            },
        }
    ],
}

In [ ]:
aoi_gdf = gpd.GeoDataFrame.from_features(aoi["features"], crs="EPSG:4326").to_crs(
    gdf.crs
)

In [ ]:
gdf_filter = gdf[gdf.intersects(aoi_gdf.geometry[0])]
gdf_filter.head()

## Visualize final results

In [ ]:
gdf_filter_plot = gdf_filter.to_crs(da.rio.crs)

fig, ax = plt.subplots(figsize=(10, 10))
show_rgb(ax, f"Filtered ships in AOI  (n={len(gdf_filter)})")
gdf_filter_plot.plot(
    ax=ax,
    column="confidence",
    cmap="plasma",
    alpha=0.6,
    edgecolor="cyan",
    linewidth=1.0,
    legend=True,
    legend_kwds={"label": "Confidence", "shrink": 0.7},
)
plt.tight_layout()
plt.show()

## Save results

In [ ]:
gdf_filter.to_file("ships_dubai.geojson")